# NLP & representation learning: Neural Embeddings, Text Classification


To use statistical classifiers with text, it is first necessary to vectorize the text. In the first practical session we explored the **Bag of Word (BoW)** model.

Modern **state of the art** methods uses  embeddings to vectorize the text before classification in order to avoid feature engineering.

## [Dataset](https://thome.isir.upmc.fr/classes/RITAL/json_pol.json)


## "Modern" NLP pipeline

By opposition to the **bag of word** model, in the modern NLP pipeline everything is **embeddings**. Instead of encoding a text as a **sparse vector** of length $D$ (size of feature dictionnary) the goal is to encode the text in a meaningful dense vector of a small size $|e| <<< |D|$.


The raw classification pipeline is then the following:

```
raw text ---|embedding table|-->  vectors --|Neural Net|--> class
```


### Using a  language model:

How to tokenize the text and extract a feature dictionnary is still a manual task. To directly have meaningful embeddings, it is common to use a pre-trained language model such as `word2vec` which we explore in this practical.

In this setting, the pipeline becomes the following:
```
      
raw text ---|(pre-trained) Language Model|--> vectors --|classifier (or fine-tuning)|--> class
```


- #### Classic word embeddings

 - [Word2Vec](https://arxiv.org/abs/1301.3781)
 - [Glove](https://nlp.stanford.edu/projects/glove/)


- #### bleeding edge language models techniques (see next)

 - [UMLFIT](https://arxiv.org/abs/1801.06146)
 - [ELMO](https://arxiv.org/abs/1802.05365)
 - [GPT](https://blog.openai.com/language-unsupervised/)
 - [BERT](https://arxiv.org/abs/1810.04805)






### Goal of this session:

1. Train word embeddings on training dataset
2. Tinker with the learnt embeddings and see learnt relations
3. Tinker with pre-trained embeddings.
4. Use those embeddings for classification
5. Compare different embedding models

## STEP 0: Loading data

In [ ]:
import json
from collections import Counter

# Loading json
file = './datasets/json_pol.json'
with open(file,encoding="utf-8") as f:
    data = json.load(f)


# Quick Check
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])

## Word2Vec: Quick Recap

**[Word2Vec](https://arxiv.org/abs/1301.3781) is composed of two distinct language models (CBOW and SG), optimized to quickly learn word vectors**


given a random text: `i'm taking the dog out for a walk`



### (a) Continuous Bag of Word (CBOW)
    -  predicts a word given a context
    
maximizing `p(dog | i'm taking the ___ out for a walk)`
    
### (b) Skip-Gram (SG)               
    -  predicts a context given a word
    
 maximizing `p(i'm taking the out for a walk | dog)`



   

## STEP 1: train a language model (word2vec)

Gensim has one of [Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) fastest implementation.


### Train:

In [ ]:
import gensim
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

text = [t.split() for t,p in data]

# # the following configuration is the default configuration
# w2v = gensim.models.word2vec.Word2Vec(sentences=text,
#                                 vector_size=100, window=5,               
#                                 min_count=5,
#                                 sample=0.001, workers=3,
#                                 sg=1, hs=0, negative=5,        ### set sg to 1 to train a skip-gram model
#                                 cbow_mean=1, epochs=5)

In [ ]:
# Worth it to save the previous embedding
# w2v.save("datasets/W2v-movies.dat")

# You will be able to reload them:
w2v = gensim.models.Word2Vec.load("datasets/W2v-movies.dat")
# and you can continue the learning process if needed

## STEP 2: Test learnt embeddings

The word embedding space directly encodes similarities between words: the vector coding for the word "great" will be closer to the vector coding for "good" than to the one coding for "bad". Generally, [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) is the distance used when considering distance between vectors.

KeyedVectors have a built in [similarity](https://radimrehurek.com/gensim/models /keyedvectors.html#gensim.models.keyedvectors.BaseKeyedVectors.similarity) method to compute the cosine similarity between words

In [ ]:
# is great really closer to good than to bad ?
print("great and good:",w2v.wv.similarity("great","good"))
print("great and bad:",w2v.wv.similarity("great","bad"))

Since cosine distance encodes similarity, neighboring words are supposed to be similar. The [most_similar](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.BaseKeyedVectors.most_similar) method returns the `topn` words given a query.

In [ ]:
# The query can be as simple as a word, such as "movie"

# Try changing the word
w2v.wv.most_similar("movie",topn=5) # 5 most similar words
#w2v.wv.most_similar("awesome",topn=5)
#w2v.wv.most_similar("actor",topn=5)

But it can be a more complicated query
Word embedding spaces tend to encode much more.

The most famous exemple is: `vec(king) - vec(man) + vec(woman) => vec(queen)`

In [ ]:
# What is awesome - good + bad ?
# w2v.wv.most_similar(positive=["awesome","bad"],negative=["good"],topn=3)

w2v.wv.most_similar(positive=["actor","woman"],negative=["man"],topn=3) # do the famous exemple works for actor ?

**To test learnt "synctactic" and "semantic" similarities, Mikolov et al. introduced a special dataset containing a wide variety of three way similarities.**

**You can download the dataset [here](https://thome.isir.upmc.fr/classes/RITAL/questions-words.txt).**

In [ ]:
# out = w2v.wv.evaluate_word_analogies("datasets/questions-words.txt",case_insensitive=True)  #original semantic syntactic dataset.

In [ ]:
'''
2026-02-14 21:43:30,644 : INFO : Evaluating word analogies for top 300000 words in the model on ressources/questions-words.txt  
2026-02-14 21:43:30,787 : INFO : capital-common-countries: 3.3% (3/90)  
2026-02-14 21:43:30,898 : INFO : capital-world: 0.0% (0/71)  
2026-02-14 21:43:30,950 : INFO : currency: 0.0% (0/28)  
2026-02-14 21:43:31,574 : INFO : city-in-state: 0.0% (0/329)  
2026-02-14 21:43:32,157 : INFO : family: 35.7% (122/342)  
2026-02-14 21:43:33,656 : INFO : gram1-adjective-to-adverb: 1.6% (15/930)  
2026-02-14 21:43:34,396 : INFO : gram2-opposite: 2.2% (12/552)  
2026-02-14 21:43:37,147 : INFO : gram3-comparative: 21.4% (270/1260)  
2026-02-14 21:43:38,615 : INFO : gram4-superlative: 8.4% (59/702)  
2026-02-14 21:43:40,561 : INFO : gram5-present-participle: 16.1% (122/756)  
2026-02-14 21:43:43,179 : INFO : gram6-nationality-adjective: 3.7% (29/792)  
2026-02-14 21:43:46,812 : INFO : gram7-past-tense: 17.8% (224/1260)  
2026-02-14 21:43:49,719 : INFO : gram8-plural: 4.4% (36/812)  
2026-02-14 21:43:52,472 : INFO : gram9-plural-verbs: 28.6% (216/756)  
2026-02-14 21:43:52,474 : INFO : Quadruplets with out-of-vocabulary words: 55.6%  
2026-02-14 21:43:52,476 : INFO : NB: analogies containing OOV words were skipped from evaluation! To change this behavior, use "dummy4unknown=True"  
2026-02-14 21:43:52,477 : INFO : Total accuracy: 12.8% (1108/8680)  
'''
print(end="")

**When training the w2v models on the review dataset, since it hasn't been learnt with a lot of data, it does not perform very well.**


## Word2vec visualisation

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt

words = list(w2v.wv.index_to_key[:100])
vectors = np.array([w2v.wv[word] for word in words])

tsne = TSNE(n_components=2, random_state=0)
reduced = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 10))
for i, word in enumerate(words):
    x, y = reduced[i]
    plt.scatter(x, y)
    plt.text(x, y, word)

plt.show()

## STEP 3: Loading a pre-trained model

In Gensim, embeddings are loaded and can be used via the ["KeyedVectors"](https://radimrehurek.com/gensim/models/keyedvectors.html) class

> Since trained word vectors are independent from the way they were trained (Word2Vec, FastText, WordRank, VarEmbed etc), they can be represented by a standalone structure, as implemented in this module.

>The structure is called “KeyedVectors” and is essentially a mapping between entities and vectors. Each entity is identified by its string id, so this is a mapping between {str => 1D numpy array}.

>The entity typically corresponds to a word (so the mapping maps words to 1D vectors), but for some models, they key can also correspond to a document, a graph node etc. To generalize over different use-cases, this module calls the keys entities. Each entity is always represented by its string id, no matter whether the entity is a word, a document or a graph node.

**You can download the pre-trained word embedding with bload = False .**

In [ ]:
# from gensim.test.utils import get_tmpfile
import gensim.downloader as api
from gensim.models import KeyedVectors

bload = True
fname = "word2vec-google-news-300"
sdir = "datasets/"

if bload == True:
    wv_pre_trained = KeyedVectors.load(sdir + fname + ".dat")
else:
    wv_pre_trained = api.load(fname)
    wv_pre_trained.save(sdir + fname + ".dat")

**Perform the "synctactic" and "semantic" evaluations again. Conclude on the pre-trained embeddings.**

In [ ]:
# out = wv_pre_trained.evaluate_word_analogies("ressources/questions-words.txt",case_insensitive=True)

In [ ]:
'''
2026-02-14 23:17:43,405 : INFO : Evaluating word analogies for top 300000 words in the model on ressources/questions-words.txt
2026-02-14 23:18:00,593 : INFO : capital-common-countries: 83.2% (421/506)
2026-02-14 23:19:03,500 : INFO : capital-world: 81.3% (3552/4368)
2026-02-14 23:19:14,798 : INFO : currency: 28.5% (230/808)
2026-02-14 23:19:50,545 : INFO : city-in-state: 72.1% (1779/2467)
2026-02-14 23:19:57,740 : INFO : family: 86.2% (436/506)
2026-02-14 23:20:12,460 : INFO : gram1-adjective-to-adverb: 29.2% (290/992)
2026-02-14 23:20:25,692 : INFO : gram2-opposite: 43.5% (353/812)
2026-02-14 23:20:45,948 : INFO : gram3-comparative: 91.3% (1216/1332)
2026-02-14 23:21:03,322 : INFO : gram4-superlative: 88.0% (987/1122)
2026-02-14 23:21:21,242 : INFO : gram5-present-participle: 78.5% (829/1056)
2026-02-14 23:21:46,305 : INFO : gram6-nationality-adjective: 90.2% (1442/1599)
2026-02-14 23:22:12,385 : INFO : gram7-past-tense: 65.4% (1020/1560)
2026-02-14 23:22:32,836 : INFO : gram8-plural: 87.0% (1159/1332)
2026-02-14 23:22:45,267 : INFO : gram9-plural-verbs: 68.2% (593/870)
2026-02-14 23:22:45,270 : INFO : Quadruplets with out-of-vocabulary words: 1.1%
2026-02-14 23:22:45,274 : INFO : NB: analogies containing OOV words were skipped from evaluation! To change this behavior, use "dummy4unknown=True"
2026-02-14 23:22:45,274 : INFO : Total accuracy: 74.0% (14307/19330)
'''
print(end="")
# On evite 5min d'execution 

## STEP 4:  sentiment classification

In the previous practical session, we used a bag of word approach to transform text into vectors.
Here, we propose to try to use word vectors (previously learnt or loaded).


### <font color='green'> Since we have only word vectors and that sentences are made of multiple words, we need to aggregate them. </font>


### (1) Vectorize reviews using word vectors:

Word aggregation can be done in different ways:

- Sum
- Average
- Min/feature
- Max/feature

#### a few pointers:

- `w2v.wv.vocab` is a `set()` of the vocabulary (all existing words in your model)
- `np.minimum(a,b) and np.maximum(a,b)` respectively return element-wise min/max

In [ ]:
# Mélange des données
np.random.seed(0)
np.random.shuffle(data)

# Ratio train / test
split_ratio = 0.8
split_id = int(len(data) * split_ratio)

train = data[:split_id]
test = data[split_id:]


def vectorize(text, model, mean=True):
    """
    This function should vectorize one review

    input: str
    output: np.array(float)
    """
    
    vectors = [model[word] for word in text if word in model]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    if mean:
        return np.mean(vectors, axis=0)
    else:
        return np.sum(vectors, axis=0)

In [ ]:
y = [pol for text, pol in train]
X = [vectorize(text.split(), wv_pre_trained) for text, pol in train]
X_test = [vectorize(text.split(), wv_pre_trained) for text, pol in test]
y_test = [pol for text, pol in test]

# let's see what a review vector looks like.
print(X[0])

### (2) Train a classifier
as in the previous practical session, train a logistic regression to do sentiment classification with word vectors



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Scikit Logistic Regression

lr = LogisticRegression().fit(X, y)

pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, pred)
print(f"{accuracy = :.2%}")

performance should be worst than with bag of word (~80%). Sum/Mean aggregation does not work well on long reviews (especially with many frequent words). This adds a lot of noise.

## Try answering the following questions:

- Which word2vec model works best: skip-gram or cbow
- Do pretrained vectors work best than those learnt on the train dataset ?

cbow (~77%) is the worst, skip-gram (~83%) and pretrained vectors (~83%) are equivalent.

## Evaluate the same pipeline on speaker ID task (Chirac/Mitterrand)

In [ ]:
import codecs
import re
from sklearn.model_selection import train_test_split

dir_path = "../datasets/"


# Chargement des données:
def load_pres(fname):
    alltxts = []
    alllabs = []
    s = codecs.open(fname, "r", "utf-8")  # pour régler le codage
    while True:
        txt = s.readline()
        if (len(txt)) < 5:
            break
        #
        lab = re.sub(r"<[0-9]*:[0-9]*:(.)>.*", "\\1", txt)
        txt = re.sub(r"<[0-9]*:[0-9]*:.>(.*)", "\\1", txt)
        if lab.count("M") > 0:
            alllabs.append(-1)
        else:
            alllabs.append(1)

        alltxts.append(txt)
    return alltxts, alllabs


fname = dir_path + "corpus.tache1.learn.utf8"
alltxts, y = load_pres(fname)

In [ ]:
X = [vectorize(txt.split(), wv_pre_trained) for txt in alltxts]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=split_ratio, random_state=42
)


lr = LogisticRegression().fit(X_train, y_train)
pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, pred)
f1_score_ = f1_score(y_test, pred)
auc = roc_auc_score(y_test, pred)

print(f"{accuracy = :.3%}")
print(f"f1_score = {f1_score_:.3%}")
print(f"{auc = :.3%}")


**(Bonus)** To have a better accuracy, we could try two things:
- Better aggregation methods (weight by tf-idf ?)
- Another word vectorizing method such as [fasttext](https://radimrehurek.com/gensim/models/fasttext.html)
- A document vectorizing method such as [Doc2Vec](https://radimrehurek.com/gensim/models/doc2vec.html)

# TF-IDF 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(alltxts)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=split_ratio, random_state=42
)


svm_clf = LogisticRegression(random_state=42).fit(X_train, y_train)
pred = svm_clf.predict(X_test)

accuracy = accuracy_score(y_test, pred)
f1_score_ = f1_score(y_test, pred)
auc = roc_auc_score(y_test, pred)

print(f"{accuracy = :.3%}")
print(f"f1_score = {f1_score_:.3%}")
print(f"{auc = :.3%}")

# Doc2Vec

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


# 1. Tokenisation et création des TaggedDocuments
def tokenize_text(text_list):
    # On garde uniquement les caractères alphanumériques
    return [re.findall(r"\w+", text.lower()) for text in text_list]


tokenized_texts = tokenize_text(alltxts)
tagged_data = [
    TaggedDocument(words=doc, tags=[i]) for i, doc in enumerate(tokenized_texts)
]

# 2. Initialisation et entraînement du modèle
vector_size = 100
model = Doc2Vec(vector_size=vector_size, window=5, min_count=1, workers=4, epochs=50)

# Construction du vocabulaire
model.build_vocab(tagged_data)

# Entraînement
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)


def get_vectors(model, corpus):
    return np.array([model.infer_vector(doc) for doc in corpus])


X_vec = get_vectors(model, tokenized_texts)

In [ ]:
from sklearn.svm import LinearSVC


X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, train_size=split_ratio, random_state=42
)

svm_clf = LinearSVC(random_state=42).fit(X_train, y_train)
pred = svm_clf.predict(X_test)

accuracy = accuracy_score(y_test, pred)
f1_score_ = f1_score(y_test, pred)
auc = roc_auc_score(y_test, pred)

print(f"{accuracy = :.3%}")
print(f"f1_score = {f1_score_:.3%}")
print(f"{auc = :.3%}")

# FastText

### Test without pre-trained model

In [ ]:
from gensim.models import FastText

# Prepare the data (using the 'text' variable from your previous movie review step)
# text = [t.split() for t,p in data]

# Initialize and train the model
# size=100 (vector dimension), window=5 (context), min_count=5 (ignore rare words)
# ft_model = FastText(sentences=text, 
#                     vector_size=100, 
#                     window=5, 
#                     min_count=5, 
#                     workers=4, 
#                     sg=1) # Using Skip-Gram

# Save the model
# ft_model.save("datasets/fasttext-movies.model")

ft_model = FastText.load("datasets/fasttext-movies.model")

In [27]:
def vectorize_ft(text_list, model):
    """
    Vectorizes a list of words using FastText.
    """
    # FastText wv[] will handle OOV automatically by synthesizing from n-grams
    vectors = [model[word] for word in text_list]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

# Vectorize the movie reviews
X_ft = [vectorize_ft(t.split(), ft_model.wv) for t in alltxts]

X_train, X_test, y_train, y_test = train_test_split(
    X_ft, y, train_size=split_ratio, random_state=42
)

# Train Logistic Regression
lr_ft = LogisticRegression(random_state=42).fit(X_train, y_train)
pred_ft = lr_ft.predict(X_test)

# print(f"FastText Accuracy: {accuracy_score(y_test, pred_ft):.2%}")

accuracy = accuracy_score(y_test, pred_ft)
f1_score_ = f1_score(y_test, pred_ft)
auc = roc_auc_score(y_test, pred_ft)

print(f"{accuracy = :.3%}")
print(f"f1_score = {f1_score_:.3%}")
print(f"{auc = :.3%}")

accuracy = 87.181%
f1_score = 93.119%
auc = 51.512%
